# Dhvani: free Kaggle GPU preparation

This notebook uses Kaggle's free GPU for data validation and an *attempted* F5-TTS fine-tune. GPU availability, runtime duration, and VRAM are not guaranteed. Do not upload voice recordings to a public dataset.

In [ ]:
!git clone --depth 1 https://github.com/karthik7026/dhvani-kannada-tts.git /kaggle/working/dhvani
%cd /kaggle/working/dhvani
!pip -q install -r requirements.txt
!git clone --depth 1 https://github.com/SWivid/F5-TTS.git /kaggle/working/F5-TTS
!pip -q install -e /kaggle/working/F5-TTS
!nvidia-smi

Attach a **private** Kaggle dataset named `dhvani-private-voice-data` containing `metadata.csv` and `wavs/`. Every row must have an exact transcript and status `approved`; draft transcripts are not sufficient for training.

In [ ]:
from pathlib import Path
import subprocess
data_dir = Path('/kaggle/input/dhvani-private-voice-data')
assert (data_dir / 'metadata.csv').is_file(), 'Attach the private voice dataset first.'
subprocess.run(['python', 'training/export_f5_manifest.py', str(data_dir), '/kaggle/working/metadata_f5.csv'], check=True)
print('Reviewed manifest is valid.')

In [ ]:
%cd /kaggle/working/F5-TTS
!python src/f5_tts/train/datasets/prepare_csv_wavs.py /kaggle/working/metadata_f5.csv data/dhvani_kn_custom --pretrain
# Start with a conservative batch size for a free 16 GB GPU.
!f5-tts_finetune-cli --exp_name F5TTS_v1_Base --dataset_name dhvani_kn_custom --tokenizer custom --tokenizer_path data/dhvani_kn_custom/vocab.txt --finetune --epochs 10 --batch_size_per_gpu 400 --batch_size_type frame --max_samples 16 --grad_accumulation_steps 8 --save_per_updates 250 --keep_last_n_checkpoints 3

If the run reports out-of-memory, lower `batch_size_per_gpu` to 250 and restart the final cell. Download the resulting checkpoint privately; never commit it or raw voice clips to GitHub.